# Contours & Shape Detection

---

## Learning Objectives

Students will learn:
- What contours and hierarchies are in computer vision.
- How to extract mathematical boundaries (`cv2.findContours`).
- How to draw those boundaries on the screen (`cv2.drawContours`).
- How to simplify shapes (`cv2.approxPolyDP`).
- How to teach the AI to count corners and logically identify shapes (e.g., triangle vs. circle).

---

## Prerequisites
- Deep understanding of Image Thresholding (creating Binary images).
- Familiarity with the OpenCV coordinate system.

---

## Dataset / Assets Used
- `../data/images/shapes.png`


## Import Libraries
Let's start by importing the necessary libraries.

In [ ]:
import cv2
import numpy as np

## Verify OpenCV Installation

In [ ]:
print(f"OpenCV Version: {cv2.__version__}")

## The Core Concepts of Contours

- **What is a Contour?** A contour is a continuous curve joining all points (along the boundary) that have the same color or intensity.
- **The Binary Requirement:** Contours work best on strictly **Binary (Black and White) images**. You must use Thresholding or Canny Edge Detection *before* looking for contours.
- **Hierarchy:** Stores relationships. If you have a small square inside a big square, the big one is the "Parent" and the small one is the "Child."

**Retrieval Modes:**
- `cv2.RETR_EXTERNAL`: Returns **only** the outermost shapes.
- `cv2.RETR_TREE`: Returns **all** shapes and maps out the full hierarchy.

**Approximation Methods:**
- `cv2.CHAIN_APPROX_SIMPLE`: Simplifies the shape, storing only the exact corner points.
- `cv2.CHAIN_APPROX_NONE`: Stores every single pixel along the boundary (memory heavy).


## Finding and Drawing Contours

This script extracts boundaries from an image and draws a bright green line over them.


In [ ]:
image_path = "../data/images/shapes.png"
# 1. Load the color image to draw on later
image = cv2.imread(image_path)

if image is not None:
    # 2. Convert to Grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 3. Create a Binary Image (Thresholding)
    _, thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
    
    # 4. Find the Contours
    # Returns the list of contour points, and the hierarchy map
    contours, hierarchy = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    
    # 5. Draw the Contours onto the ORIGINAL color image
    # -1 means "Draw ALL contours". (Passing 0 would only draw the first one).
    cv2.drawContours(image, contours, -1, (0, 255, 0), 3)
    
    cv2.imshow("Contours Drawn", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print(f"Error: Image not found at {image_path}")

## Detecting Specific Shapes (`cv2.approxPolyDP`)

How does a computer know if a shape is a triangle or a rectangle? **It counts the corners.**
Because raw contours have dozens of microscopic jagged points, we must mathematically simplify the curve using `approxPolyDP`.

**The Math (Epsilon):** Epsilon dictates how closely the simplified shape must match the original raw contour.
- `epsilon = 0.01 * cv2.arcLength(contour, True)`
- We multiply `0.01` by the `arcLength` (perimeter). Small multipliers create accurate shapes, large multipliers create rough approximations.


In [ ]:
if image is not None:
    # Re-read to reset the drawing
    image = cv2.imread(image_path)
    
    for contour in contours:
        # 1. Calculate the perimeter (True means the shape is a closed loop)
        perimeter = cv2.arcLength(contour, True)
        
        # 2. Simplify the shape to its core corners
        approx = cv2.approxPolyDP(contour, 0.01 * perimeter, True)
        
        # 3. Count the corners
        corners = len(approx)
        
        # 4. Determine shape based on corner count
        if corners == 3:
            shape_name = "Triangle"
        elif corners == 4:
            shape_name = "Rectangle/Square"
        elif corners == 5:
            shape_name = "Pentagon"
        elif corners > 5:
            shape_name = "Circle"
        else:
            shape_name = "Unknown"
            
        # 5. Get a coordinate to place our text label
        # approx.ravel() flattens the multi-dimensional array
        x = approx.ravel()[0]
        y = approx.ravel()[1] - 10 
        
        # 6. Put the label on the image
        cv2.putText(image, shape_name, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    cv2.imshow("Shape Detector", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

## Common Mistakes
> **Warning**  
> - Trying to run `cv2.findContours` on a regular BGR color image. It will crash. It needs a binary 1-channel image.
> - Forgetting to use `.copy()` when drawing contours if you need to keep the original image pristine.
> - Setting the epsilon multiplier too high in `approxPolyDP`, which might make a circle look like a triangle to the computer.


## Key Takeaways
- Thresholding is the critical first step before contour detection.
- `cv2.findContours` finds the math, `cv2.drawContours` makes it visible.
- `cv2.approxPolyDP` is required to simplify curves so you can accurately count corners.


## Practice Exercises
1. Change the retrieval mode to `cv2.RETR_EXTERNAL` on an image that has shapes inside of shapes, and observe how the inner shapes are ignored.
2. Change the epsilon multiplier in `approxPolyDP` to `0.1` and see how the detection accuracy degrades.
3. Draw only the 2nd contour in the list (instead of `-1` for all) by passing `1` to `drawContours`.
4. Add logic to differentiate between a Rectangle and a Square by calculating the aspect ratio (width divided by height).
5. Modify the Shape Detector loop to draw the text in different colors (e.g., Blue for Triangle, Green for Rectangle, Red for Circle).


## Next Notebook
In our final phase, we'll combine our knowledge of grayscaling, slicing, and shapes to use pre-trained AI models to detect complex features like Human Faces!

👉 Proceed to: **[08_Face_and_Object_Detection.ipynb](./08_Face_and_Object_Detection.ipynb)**
